In [1]:

import os
import shutil


from google.colab import userdata
USER = "Koubra-gaby"
#TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

if os.path.exists(REPO_NAME):
    print(f"Le dossier '{REPO_NAME}' existe déjà. Suppression en cours...")
    shutil.rmtree(REPO_NAME)
# Cloner le dépôt privé
!git clone https://{USER}:@github.com/{USER}/{REPO_NAME}.git


Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 535, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 535 (delta 107), reused 73 (delta 29), pack-reused 348 (from 1)
Receiving objects: 100% (535/535), 24.54 MiB | 28.49 MiB/s, done.
Resolving deltas: 100% (271/271), done.


In [9]:
%%capture
# 1. Installer et activer Git LFS
!apt-get install git-lfs
!git lfs install

# 2. Forcer la récupération des vrais fichiers LFS
!git lfs pull


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 0s (13.0 MB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Updated Git hooks.
Git LFS initialized.


In [2]:
# via python & Jupyter
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('update_shu'))

## training fastext model for chadian arabic corpus

In [5]:
%%capture
!pip install gensim
!pip install nltk


In [6]:
## Importing libraries
from gensim.models import FastText
from gensim.test.utils import common_texts

In [ ]:
import re
from nltk.tokenize import word_tokenize


def preparer_sentences_entrainer_fastext(fichier1, fichier2, file_to_save, lang):
    sentences = []
    if lang == "fr":
      patern = r"[^\w\s'-]"
    else :
      patern = r"[^\w\s']"
    for fichier in [fichier1, fichier2]:
        with open(fichier, "r", encoding="utf-8-sig") as f:
            for ligne in f:
                ligne = ligne.strip()

                if ligne:
                    # Remplacer les ponctuations par des espaces,
                    # sauf les apostrophes et les tirets
                    ligne = re.sub(
                        patern,
                        " ",
                        ligne,
                        flags=re.UNICODE
                    )

                    # Tokenisation mot par mot
                    tokens = word_tokenize(
                        ligne,
                        preserve_line=True
                    )

                    if tokens:
                        sentences.append(tokens)

      ## training fastext
    model = FastText(
      sentences,
      vector_size=1024,
      window=5,
      min_count=1,
      epochs=10,
      workers=4,
      sg=1,
  )
    model.wv.save_word2vec_format(file_to_save)

    print("Model trained and savec successfully")

    return model,sentences

In [ ]:
## Loading the corpus then tokenizing it
data_path = "/content/Chadian_arabic-update/Data/full_dataset/"
model_arabic,sentences_arabic = preparer_sentences_entrainer_fastext(data_path + "arabe_tchad3.txt",data_path + "arabic_not_bible.txt", data_path + "arabic_embedding.vec", lang = "shu")
sentences_arabic

Model trained and savec successfully


[['Fi',
  'l',
  'bidaaya',
  'Allah',
  'khalag',
  'al',
  'samaawaat',
  'wa',
  'l',
  'ard'],
 ['Wa',
  'l',
  'ard',
  'faadiye',
  'Hi',
  'ma',
  'indaha',
  'cheyy',
  'foogha',
  'Wa',
  'l',
  'dalaam',
  "gaa'id",
  'yikhatti',
  'al',
  'almi',
  'al',
  'khariig',
  'wa',
  'Ruuh',
  'Allah',
  "gaa'id",
  'yuhuum',
  'fi',
  'l',
  'almi'],
 ['Wa',
  'Allah',
  'al',
  'Rabb',
  'gaal',
  'Khalli',
  'yukuun',
  'nuur',
  'Wa',
  'khalaas',
  'al',
  'nuur',
  'kaan'],
 ['Wa',
  'Allah',
  'al',
  'Rabb',
  'chaaf',
  'kadar',
  'al',
  'nuur',
  'da',
  'sameh',
  'Wa',
  'farag',
  'al',
  'nuur',
  'min',
  'al',
  'dalaam'],
 ['Wa',
  'samma',
  'al',
  'nuur',
  'nahaar',
  'wa',
  'l',
  'dalaam',
  'leel',
  'Wa',
  'kaan',
  'achiiye',
  'wa',
  'kaan',
  'fajur',
  'Da',
  'l',
  'yoom',
  'al',
  'awwal'],
 ['Wa',
  'Allah',
  'al',
  'Rabb',
  'gaal',
  'Khalli',
  'tukuun',
  'faraga',
  'been',
  'al',
  'almi',
  'wa',
  'tifaarig',
  'al',
  'almi',
  'min

In [ ]:
## Finding most similar words

print("Words similar to 'ana':")

similar_words = model_arabic.wv.most_similar(
    "ana",
    topn=10
)

for word, score in similar_words:
    print(word, ":", round(score, 4))

Words similar to 'ana':
Ana : 0.7406
nisiitni : 0.7281
ni : 0.72
nafsiana : 0.7186
zaati : 0.7139
nidimt : 0.7126
nisatni : 0.71
naadeetuuni : 0.6997
Dossoni : 0.6978
zalamni : 0.697


In [3]:
%cd /content/Chadian_arabic-update

/content/Chadian_arabic-update


In [ ]:
from models.extraction_french_emb import *
embeddings = extract_nllb_word_embeddings(
    input_file="/content/Chadian_arabic-update/Data/vocabulaire_unique_french.txt",
    output_file="/content/Chadian_arabic-update/Data/french_nllb_espace.vec"
)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] M2M100Model LOAD REPORT from: facebook/nllb-200-distilled-600M
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extraction des embeddings:   0%|          | 0/23685 [00:00<?, ?it/s]

Extraction terminée : 23685 mots
Dimension des embeddings : 1024
Fichier enregistré : /content/Chadian_arabic-update/Data/french_nllb_espace.vec


In [10]:
with open("/content/Chadian_arabic-update/Data/french_nllb_espace.vec", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

23685 1024
A -0.03778076 -0.00822449 0.01852417 0.00742340 0.01747131 -0.02383423 0.02618408 -0.02221680 -0.01890564 -0.07098389 -0.02369690 -0.02026367 -0.03967285 -0.00908661 -0.05880737 0.01853943 0.08966064 0.04403687 0.07244873 0.03005981 0.12951660 -0.07727051 0.01184082 -0.02563477 0.03010559 0.24182129 -0.20861816 0.15368652 0.01383209 -0.12988281 -0.08093262 -0.05004883 -0.08410645 -0.02044678 -0.02886963 0.13549805 -0.10888672 0.03683472 -0.12634277 -0.11822510 -0.04492188 -0.07965088 -0.32446289 0.04174805 -0.12634277 0.39697266 -0.02671814 -0.18444824 0.14721680 -0.27197266 0.18493652 0.10540771 0.10961914 -0.06970215 0.06866455 -0.02821350 0.08215332 -0.18420410 0.16174316 0.14953613 -0.06787109 0.08331299 -0.01028442 -0.06872559 0.16503906 -0.17907715 0.09240723 0.04901123 -0.07373047 0.01476288 -0.02258301 -0.24829102 0.19970703 -0.14611816 0.06134033 0.23059082 0.04528809 -0.02680969 -0.03759766 -0.09393311 0.02296448 -0.03099060 0.12481689 -0.00337601 -0.23901367 0.131

In [11]:
from gensim.models import KeyedVectors

vectors = KeyedVectors.load_word2vec_format(
    "/content/Chadian_arabic-update/Data/french_nllb_espace.vec",
    binary=False
)

print("Vocabulaire :", len(vectors))
print("Dimension :", vectors.vector_size)

Vocabulaire : 23685
Dimension : 1024


In [15]:
word = "salutations"

similar_words = vectors.most_similar(
    word,
    topn=10
)

for similar_word, score in similar_words:
    print(f"{similar_word:<20} {score:.4f}")

salutation           0.7963
salut                0.7086
lamentations         0.6458
recommandations      0.6098
invitations          0.6054
adorations           0.6006
consultations        0.5869
divinations          0.5864
dispensations        0.5816
agitations           0.5782


In [ ]:
!git clone https://github.com/facebookresearch/MUSE.git

Cloning into 'MUSE'...
remote: Enumerating objects: 239, done.
remote: Total 239 (delta 0), reused 0 (delta 0), pack-reused 239 (from 1)
Receiving objects: 100% (239/239), 215.77 KiB | 35.96 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [ ]:
%cd /content/MUSE

from pathlib import Path

path = Path("src/utils.py")
code = path.read_text()

code = code.replace(
    "inspect.getargspec(optim_fn.__init__)[0]",
    "inspect.getfullargspec(optim_fn.__init__).args"
)

path.write_text(code)

print("Modification appliquée.")

/content/MUSE
Modification appliquée.


In [ ]:
file_path = "/content/MUSE/src/trainer.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(path)",
    "torch.load(path, weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Correction appliquée.")

Correction appliquée.


In [ ]:
!pip install -U faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 79.3 MB/s  0:00:01


In [ ]:
!python unsupervised.py \
    --src_lang shu \
    --tgt_lang fr \
    --src_emb /content/Chadian_arabic-update/Data/arabic_embedding.vec \
    --tgt_emb /content/Chadian_arabic-update/Data/french_nllb_espace.vec \
    --emb_dim 1024 \
    --n_refinement 10 \
    --n_epochs 10 \
    --dis_most_frequent 20000 \
    --dico_eval /content/Chadian_arabic-update/Data/dico_eval.txt

INFO - 11/05/26 17:50:38 - 0:00:00 - ============ Initialized logger ============
INFO - 11/05/26 17:50:38 - 0:00:00 - adversarial: True
                                     batch_size: 32
                                     cuda: True
                                     dico_build: S2T
                                     dico_eval: /content/Chadian_arabic-update/Data/dico_eval.txt
                                     dico_max_rank: 15000
                                     dico_max_size: 0
                                     dico_method: csls_knn_10
                                     dico_min_size: 0
                                     dico_threshold: 0
                                     dis_clip_weights: 0
                                     dis_dropout: 0.0
                                     dis_hid_dim: 2048
                                     dis_input_dropout: 0.1
                                     dis_lambda: 1
                                     dis_layers: 2
 

# Tester le modele entrainer

In [ ]:
import io
import numpy as np

In [ ]:
def load_vec(emb_path, nmax=50000):
    vectors = []
    word2id = {}
    with io.open(emb_path, 'r', encoding='utf-8', newline='\n', errors='ignore') as f:
        next(f)
        for i, line in enumerate(f):
            word, vect = line.rstrip().split(' ', 1)
            vect = np.fromstring(vect, sep=' ')
            assert word not in word2id, 'word found twice'
            vectors.append(vect)
            word2id[word] = len(word2id)
            if len(word2id) == nmax:
                break
    id2word = {v: k for k, v in word2id.items()}
    embeddings = np.vstack(vectors)
    return embeddings, id2word, word2id

## Get nearest neighbors

In [ ]:
src_path = '/content/MUSE/dumped/debug/dtxwjv76ku/vectors-shu.txt'
tgt_path = '/content/MUSE/dumped/debug/dtxwjv76ku/vectors-fr.txt'
nmax = 50000  # maximum number of word embeddings to load

src_embeddings, src_id2word, src_word2id = load_vec(src_path, nmax)
tgt_embeddings, tgt_id2word, tgt_word2id = load_vec(tgt_path, nmax)

In [ ]:
def get_nn(word, src_emb, src_id2word, tgt_emb, tgt_id2word, K=5):
    print("Nearest neighbors of \"%s\":" % word)
    word2id = {v: k for k, v in src_id2word.items()}
    word_emb = src_emb[word2id[word]]
    scores = (tgt_emb / np.linalg.norm(tgt_emb, 2, 1)[:, None]).dot(word_emb / np.linalg.norm(word_emb))
    k_best = scores.argsort()[-K:][::-1]
    for i, idx in enumerate(k_best):
        print('%.4f - %s' % (scores[idx], tgt_id2word[idx]))

In [ ]:
# printing nearest neighbors in the source space
src_word = 'ana'
get_nn(src_word, src_embeddings, src_id2word, src_embeddings, src_id2word, K=5)

Nearest neighbors of "ana":
1.0000 - ana
0.8043 - nisiitni
0.7972 - Ana
0.7945 - nafsiana
0.7927 - zaati


In [ ]:
# printing nearest neighbors in the target space
src_word = 'moi'
get_nn(src_word, tgt_embeddings, tgt_id2word, src_embeddings, src_id2word, K=5)

Nearest neighbors of "moi":
0.1085 - hayaatha
0.1040 - hayne
0.1037 - khadamtaha
0.1026 - nukhuttaha
0.1026 - badalha


# Projection vectorielle

In [ ]:
%%capture
!pip install -q gensim transformers scipy sentencepiece

### Sauvegarder le modèle, le tokenizer et la transformation

In [ ]:

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


def add_muse_words_to_nllb(
    muse_file,
    tokenizer,
    model,
    language="shu_Latn"
):
    """
    Ajoute les mots arabes absents du tokenizer NLLB
    et initialise leurs embeddings avec les vecteurs MUSE,
    sans aucune transformation des vecteurs.

    Parameters
    ----------
    muse_file : str
        Fichier .vec contenant les mots et leurs vecteurs MUSE.
    tokenizer : AutoTokenizer
        Tokenizer NLLB.
    model : AutoModel ou AutoModelForSeq2SeqLM
        Modèle NLLB.
    language : str
        Code de langue utilisé pour le tokenizer.

    Returns
    -------
    tokenizer, model, report
    """

    # 1. Lire les vecteurs MUSE
    muse_vectors = {}

    with open(muse_file, "r", encoding="utf-8") as f:

        header = f.readline().strip().split()
        expected_vocab_size = int(header[0])
        muse_dim = int(header[1])

        for line in tqdm(f, desc="Lecture des vecteurs MUSE"):

            parts = line.rstrip().split()

            if len(parts) != muse_dim + 1:
                continue

            word = parts[0]
            vector = np.asarray(
                parts[1:],
                dtype=np.float32
            )

            if word not in muse_vectors:
                muse_vectors[word] = vector

    print("Mots MUSE chargés :", len(muse_vectors))
    print("Dimension MUSE :", muse_dim)

    # 2. Vérifier les dimensions
    embedding_layer = model.get_input_embeddings()
    embedding_dim = embedding_layer.weight.shape[1]

    if muse_dim != embedding_dim:
        raise ValueError(
            f"Dimensions incompatibles : MUSE={muse_dim}, "
            f"NLLB={embedding_dim}"
        )

    # 3. Identifier les mots déjà présents
    existing_tokens = set(tokenizer.get_vocab().keys())

    new_words = [
        word for word in muse_vectors
        if word not in existing_tokens
    ]

    existing_words = [
        word for word in muse_vectors
        if word in existing_tokens
    ]

    print("Mots déjà présents :", len(existing_words))
    print("Nouveaux mots à ajouter :", len(new_words))

    if not new_words:
        return tokenizer, model, {
            "added": 0,
            "already_present": len(existing_words)
        }

    # 4. Ajouter les mots au tokenizer
    old_vocab_size = len(tokenizer)

    added_count = tokenizer.add_tokens(new_words)

    if added_count != len(new_words):
        raise RuntimeError(
            "Le nombre de tokens ajoutés ne correspond pas "
            "au nombre de mots absents."
        )

    # 5. Agrandir la matrice d'embeddings
    model.resize_token_embeddings(len(tokenizer))

    # 6. Copier les vecteurs MUSE sans modification
    embedding_layer = model.get_input_embeddings()

    with torch.no_grad():

        for word in tqdm(new_words, desc="Initialisation NLLB"):

            token_id = tokenizer.convert_tokens_to_ids(word)

            muse_vector = torch.from_numpy(
                muse_vectors[word]
            ).to(
                device=embedding_layer.weight.device,
                dtype=embedding_layer.weight.dtype
            )

            embedding_layer.weight[token_id].copy_(
                muse_vector
            )

    # 7. Vérification
    print("\nTerminé !")
    print("Ancienne taille :", old_vocab_size)
    print("Nouvelle taille :", len(tokenizer))
    print("Nombre de mots ajoutés :", added_count)

    report = {
        "added": added_count,
        "already_present": len(existing_words),
        "old_vocab_size": old_vocab_size,
        "new_vocab_size": len(tokenizer),
        "embedding_dimension": embedding_dim
    }

    return tokenizer, model, report


In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = "nllb_chadian_arabic_muse_aligned"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    #src_lang="fra_Latn"
)

model = AutoModel.from_pretrained(MODEL_NAME)

tokenizer, model, report = add_muse_words_to_nllb(
    muse_file='/content/MUSE/dumped/debug/dtxwjv76ku/vectors-shu.txt',
    tokenizer=tokenizer,
    model=model
)

print(report)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

[transformers] M2M100Model LOAD REPORT from: facebook/nllb-200-distilled-600M
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lecture des vecteurs MUSE: 0it [00:00, ?it/s]

Mots MUSE chargés : 36426
Dimension MUSE : 1024
Mots déjà présents : 690
Nouveaux mots à ajouter : 35736


Initialisation NLLB:   0%|          | 0/35736 [00:00<?, ?it/s]


Terminé !
Ancienne taille : 256204
Nouvelle taille : 291940
Nombre de mots ajoutés : 35736
{'added': 35736, 'already_present': 690, 'old_vocab_size': 256204, 'new_vocab_size': 291940, 'embedding_dimension': 1024}


In [ ]:
model.push_to_hub(OUTPUT_DIR)
tokenizer.push_to_hub(OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ipprfaf/model.safetensors:   1%|          | 16.0MB / 2.61GB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpm46tczhe/tokenizer.json:  62%|######1   | 24.0MB / 38.9MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/Koubra-Gaby/nllb_chadian_arabic_muse_aligned/commit/74062e88da162e904d49db291ddd688aeaac7638', commit_message='Upload tokenizer', commit_description='', oid='74062e88da162e904d49db291ddd688aeaac7638', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Koubra-Gaby/nllb_chadian_arabic_muse_aligned', endpoint='https://huggingface.co', repo_type='model', repo_id='Koubra-Gaby/nllb_chadian_arabic_muse_aligned'), pr_revision=None, pr_num=None)

In [ ]:
%cd /content/Chadian_arabic-update

/content/Chadian_arabic-update


In [ ]:
!git reset --soft HEAD~2


In [ ]:
!git lfs install


Updated Git hooks.
Git LFS initialized.


In [ ]:
!git lfs track "*.txt"


"*.txt" already supported


In [ ]:
!git add /content/Chadian_arabic-update/Data/vectors-fr.txt /content/Chadian_arabic-update/Data/vectors-shu.txt

In [ ]:
!git commit -m "update"

[main 6ceeedf] update
 2 files changed, 6 insertions(+)
 create mode 100644 Data/vectors-fr.txt
 create mode 100644 Data/vectors-shu.txt


In [ ]:
!git push

Uploading LFS objects: 100% (2/2), 526 MB | 0 B/s, done.
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 599 bytes | 599.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
remote: This repository moved. Please use the new location:
remote:   https://github.com/koubra-gaby/Chadian_arabic-update.git
To https://github.com/Koubra-gaby/Chadian_arabic-update.git
   72c0744..6ceeedf  main -> main
